# 验证 · CoLaR-code 训练流程(小规模 + loss 收敛)
**A100 → Run all**。小规模(200 行 · 3ep)验证官方 run.py 训练流程正确、loss 下降。真训练用 `handoff/train_colar_code.sh`。

In [ ]:
# SETUP + 小数据(transformers 4.45.2 官方 CoLaR)
import os, subprocess, torch, json, random
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run('pip -q install "transformers==4.45.2" "lightning==2.5.1.post0" "peft==0.15.2" "omegaconf==2.3.0" "numpy>=2.0,<2.3" sentencepiece accelerate', shell=True)
subprocess.run('pip -q install --force-reinstall --no-deps "huggingface_hub==0.34.4"', shell=True)
for r,u in [("colar","https://github.com/xiaomi-research/colar.git"),("lrm","https://github.com/ruijiezh67/LRM_colab_tasks.git")]:
    if not os.path.exists(f"/content/{r}/.git"): subprocess.run(f"git clone -q {u} /content/{r}", shell=True)
WS="/content/ws"; os.makedirs(f"{WS}/models/llms", exist_ok=True); os.makedirs(f"{WS}/datasets/text_reasoning/coding_mix", exist_ok=True); os.environ["WS"]=WS
LL=f"{WS}/models/llms/Llama-3.2-1B-Instruct"
if not os.path.exists(LL+"/config.json"): subprocess.run(f'huggingface-cli download unsloth/Llama-3.2-1B-Instruct --local-dir {LL} --exclude "original/*"', shell=True, check=True)
GSM="/content/colar_hf/logs/colar/qsa-gsm/colar-final/checkpoints/colar_best.ckpt"
if not os.path.exists(GSM): subprocess.run('huggingface-cli download AlbertTan/CoLaR logs/colar/qsa-gsm/colar-final/checkpoints/colar_best.ckpt --local-dir /content/colar_hf', shell=True, check=True)
os.environ["GSM"]=GSM
tr=json.load(open("/content/lrm/code_real_ladder/colar_train.json")); random.seed(0); random.shuffle(tr); tr=tr[:200]
va=json.load(open("/content/lrm/code_real_ladder/colar_val.json"))[:40]
for nm,dd in [("train",tr),("val",va),("test",va)]: json.dump(dd, open(f"{WS}/datasets/text_reasoning/coding_mix/{nm}.json","w"), ensure_ascii=False, indent=1)
print("SETUP OK | 验证子集 train",len(tr),"val",len(va))

In [ ]:
# 小规模训练 + loss 收敛画图
import os
os.system("pkill -f 'run.py' 2>/dev/null; sleep 2")
WS=os.environ["WS"]; GSM=os.environ["GSM"]
!cd /content/colar && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python -u run.py --model=colar --dataset=qsa --devices=0 --workspace_path={WS} --load_ckpt_path={GSM} --log_suffix=verify --seed=0 dataset_name=coding_mix model_id=Llama-3.2-1B-Instruct batch_size=4 accumulate_grad_batches=2 max_epochs=3 check_val_every_n_epoch=1 2>&1 | tee /content/train.log

import re, glob, matplotlib.pyplot as plt
def show_loss(logpath, title):
    txt = open(logpath, encoding="utf-8", errors="ignore").read()
    ls = [float(x) for x in re.findall(r"(?:'loss'|train_loss|loss)[=:'\s]+([0-9]+\.[0-9]+)", txt)]
    ls = [x for x in ls if x < 1e4]
    if len(ls) >= 2:
        plt.figure(figsize=(6,3)); plt.plot(ls, marker="."); plt.title(title+" · loss"); plt.xlabel("log step"); plt.ylabel("loss"); plt.grid(alpha=.3); plt.show()
        print(f"loss: {ls[0]:.3f} -> {ls[-1]:.3f} ({len(ls)} 点)")
        print("✅ [PASS] 训练流程正确: loss 下降(收敛趋势)" if ls[-1] < ls[0] else "⚠ [WARN] 跑通但 loss 没降 → 查 lr/数据/配方")
    else:
        print("⚠ 没抓到 loss 序列 → 看上面日志确认训练是否真启动")

show_loss("/content/train.log", "CoLaR-code")